# NeuroXVocal — Evaluation on ADReSSo Diagnosis-Test Set

Evaluates the trained model on the 71 held-out test patients.  
Includes **VRAM & timing monitor** for each of the 3 inference pipelines.

**Before running:** `Runtime → Change runtime type → T4 GPU`

## 0. Check GPU

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device       :', torch.cuda.get_device_name(0))
    print('Total VRAM   :', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), 'GB')

## 1. Install Dependencies

In [ ]:
%%capture
!pip install openai-whisper librosa praat-parselmouth soundfile scipy
!pip install transformers==4.45.2 sentencepiece scikit-learn joblib pandas numpy tqdm

## 2. Mount Drive & Clone Repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
if not os.path.exists('/content/NeuroXVocal'):
    !git clone https://github.com/homiyano/NeuroXVocal.git /content/NeuroXVocal
else:
    print('Repo already cloned')

## 3. Configure Paths

Set these to match your Drive layout.

In [ ]:
# ── CONFIGURE THESE ─────────────────────────────────────────────────
TEST_AUDIO_DIR = '/content/drive/MyDrive/ADReSSo/diagnosis-test/diagnosis/test-dist/audio'
MODEL_PATH     = '/content/drive/MyDrive/NeuroXVocal_results/model_fold2_best.pth'
OUTPUT_CSV     = '/content/drive/MyDrive/NeuroXVocal_results/predictions.csv'
MONITOR_CSV    = '/content/drive/MyDrive/NeuroXVocal_results/vram_monitor.csv'

# Optional: CSV with columns (patient_id, label) where 1=AD 0=CN
LABELS_CSV = None
# ────────────────────────────────────────────────────────────────────

from pathlib import Path
wavs = list(Path(TEST_AUDIO_DIR).glob('*.wav'))
print(f'Test audio files : {len(wavs)}')
print(f'Model exists     : {os.path.exists(MODEL_PATH)}')
print('Sample files     :', [w.name for w in sorted(wavs)[:5]])

## 4. VRAM Monitor Class

In [ ]:
import time
import torch

class VRAMMonitor:
    """Tracks CUDA VRAM (MB) and wall-clock time per pipeline stage."""

    def __init__(self, device):
        self.device        = device
        self.dtype         = device.type
        self._global_peak  = 0.0
        self.load_records  = []

    def current_mb(self):
        if self.dtype == 'cuda':
            return torch.cuda.memory_allocated(self.device) / 1024**2
        return 0.0

    def peak_mb(self):
        if self.dtype == 'cuda':
            return torch.cuda.max_memory_allocated(self.device) / 1024**2
        return 0.0

    def reset_peak(self):
        if self.dtype == 'cuda':
            torch.cuda.reset_peak_memory_stats(self.device)

    def sync(self):
        if self.dtype == 'cuda':
            torch.cuda.synchronize(self.device)

    def measure(self, label):
        return _Stage(self, label)

    def record_load(self, name, fn):
        """Load a model and record the VRAM delta."""
        self.sync()
        before = self.current_mb()
        obj    = fn()
        self.sync()
        after  = self.current_mb()
        delta  = after - before
        self.load_records.append({'model': name, 'before_mb': round(before, 1),
                                   'after_mb': round(after, 1), 'delta_mb': round(delta, 1)})
        print(f'  Loaded {name:<26}  before={before:6.1f} MB  after={after:6.1f} MB  delta=+{delta:.1f} MB')
        return obj

    def update_global_peak(self, v):
        if v > self._global_peak:
            self._global_peak = v

    def global_peak_mb(self):
        return self._global_peak


class _Stage:
    def __init__(self, mon, label):
        self.mon   = mon
        self.label = label
        self.vram_before = self.vram_after = self.peak = self.elapsed = 0.0

    def __enter__(self):
        self.mon.sync()
        self.mon.reset_peak()
        self.vram_before = self.mon.current_mb()
        self._t0         = time.perf_counter()
        return self

    def __exit__(self, *_):
        self.mon.sync()
        self.elapsed    = time.perf_counter() - self._t0
        self.vram_after = self.mon.current_mb()
        self.peak       = self.mon.peak_mb()
        self.mon.update_global_peak(self.peak)

print('VRAMMonitor ready.')

## 5. Load All Models (with VRAM tracking)

### 5-fix. Disable XET Download Protocol (fixes slow Wav2Vec2 / Whisper download)

In [ ]:
import os
# Disable HuggingFace XET protocol — it throttles to ~70 kB/s on Colab
os.environ['HF_HUB_DISABLE_XET'] = '1'

# Cache models to Drive so they survive session restarts
HF_CACHE_DIR = os.path.join(DRIVE_BASE, 'hf_cache')
os.makedirs(HF_CACHE_DIR, exist_ok=True)
os.environ['HF_HOME'] = HF_CACHE_DIR
print('XET disabled. HuggingFace cache →', HF_CACHE_DIR)

In [ ]:
import sys, re, tempfile, shutil, subprocess, joblib, warnings
import whisper
import pandas as pd
import numpy as np
import soundfile as sf
from pathlib import Path
from math import gcd
from scipy.signal import resample_poly
from transformers import AutoTokenizer, Wav2Vec2Model, Wav2Vec2Processor
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

warnings.filterwarnings('ignore', category=UserWarning, module='sklearn')

sys.path.insert(0, '/content/NeuroXVocal/src/train')
from models import NeuroXVocal

SCALER_FEAT = '/content/NeuroXVocal/src/inference/scaler_params_audio_features.pkl'
SCALER_EMB  = '/content/NeuroXVocal/src/inference/scaler_params_audio_emb.pkl'
DROP_COLS   = ['jitter_local','shimmer_local','formant_1_mean','formant_1_std',
               'formant_2_mean','formant_2_std','formant_3_mean','formant_3_std','class']
TEXT_MODEL  = 'microsoft/deberta-v3-base'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
mon    = VRAMMonitor(device)
print(f'Device: {device}\n')

wmodel    = mon.record_load('Whisper base',
                lambda: whisper.load_model('base'))
tokenizer = mon.record_load('DeBERTa tokenizer',
                lambda: AutoTokenizer.from_pretrained(TEXT_MODEL))
processor = mon.record_load('Wav2Vec2 processor',
                lambda: Wav2Vec2Processor.from_pretrained('facebook/wav2vec2-base-960h'))
emodel    = mon.record_load('Wav2Vec2 model',
                lambda: Wav2Vec2Model.from_pretrained('facebook/wav2vec2-base-960h').to(device).eval())

def _load_clf():
    clf = NeuroXVocal(num_audio_features=47, num_embedding_features=768,
                      text_embedding_model=TEXT_MODEL)
    sd = torch.load(MODEL_PATH, map_location=device)
    if 'module.' in list(sd.keys())[0]:
        from collections import OrderedDict
        sd = OrderedDict((k.replace('module.',''), v) for k, v in sd.items())
    clf.load_state_dict(sd)
    return clf.to(device).eval()

clf = mon.record_load('NeuroXVocal (DeBERTa)', _load_clf)

print('\n── Model Load VRAM Summary ─────────────────────────────────────')
print(f'  {"Model":<28}  {"Before":>9}  {"After":>9}  {"Delta":>9}')
print('  ' + '-'*58)
for r in mon.load_records:
    print(f'  {r["model"]:<28}  {r["before_mb"]:>8.1f}M  {r["after_mb"]:>8.1f}M  +{r["delta_mb"]:>7.1f}M')
print('─'*62)

## 6. Run Evaluation with VRAM & Timing Monitor

3 pipelines tracked per patient:
- **Pipeline 1** — Whisper transcription
- **Pipeline 2** — Wav2Vec2 audio embedding
- **Pipeline 3** — NeuroXVocal classification (DeBERTa + classifier)

> Audio feature extraction runs in a subprocess — its VRAM is not visible from this process.

In [ ]:
from tqdm import tqdm

def transcribe(wav_path):
    fp16 = torch.cuda.is_available()
    r    = wmodel.transcribe(str(wav_path), fp16=fp16)
    t    = r['text'].lower()
    return re.sub(r'[^a-zA-Z0-9\s.,!?]', '', ' '.join(t.split()))

def get_audio_features(wav_path):
    script = '/content/NeuroXVocal/src/data_extraction/extract_audio_features.py'
    tmp    = Path(tempfile.mkdtemp())
    csv    = tmp / 'f.csv'
    subprocess.run([sys.executable, script, str(wav_path.parent),
                    '--output_csv', str(csv)], capture_output=True)
    df  = pd.read_csv(csv)
    shutil.rmtree(tmp)
    row = df[df['patient_id'] == wav_path.stem]
    pid = row['patient_id'].values
    row = row.drop(columns=['patient_id'] + DROP_COLS, errors='ignore')
    row = row.apply(lambda x: x.fillna(x.mean()) if x.isnull().any() else x)
    scaled = joblib.load(SCALER_FEAT).transform(row)
    out = pd.DataFrame(scaled, columns=row.columns)
    out.insert(0, 'patient_id', pid)
    return out

def get_embeddings(wav_path):
    speech, sr = sf.read(str(wav_path), always_2d=True)
    speech     = speech.mean(axis=1).astype(np.float32)
    if sr != 16000:
        g      = gcd(sr, 16000)
        speech = resample_poly(speech, 16000 // g, sr // g).astype(np.float32)
    inp = processor(speech, sampling_rate=16000, return_tensors='pt', padding=True)
    inp = {k: v.to(device) for k, v in inp.items()}
    with torch.no_grad():
        emb = emodel(**inp).last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
    return joblib.load(SCALER_EMB).transform(emb.reshape(1,-1)).flatten()

def predict(text, feat_row, emb):
    tok  = tokenizer(text, padding='max_length', truncation=True,
                     max_length=512, return_tensors='pt')
    tok  = {k: v.to(device) for k, v in tok.items()}
    af   = torch.tensor(feat_row.drop(columns=['patient_id']).iloc[0].values.astype(float),
                        dtype=torch.float32).unsqueeze(0).to(device)
    embt = torch.tensor(emb, dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        conf = torch.sigmoid(clf(tok, af, embt)).item()
    return (1 if conf > 0.5 else 0), conf

# ── Main loop ────────────────────────────────────────────────────────
wav_files       = sorted(Path(TEST_AUDIO_DIR).glob('*.wav'))
results         = []
monitor_records = []

for wav in tqdm(wav_files, desc='Patients'):
    pid = wav.stem
    try:
        # Pipeline 1 — Whisper transcription
        with mon.measure('1. Whisper transcription') as s1:
            text = transcribe(wav)
        monitor_records.append({'patient': pid, 'pipeline': '1. Whisper transcription',
                                'vram_before_mb': round(s1.vram_before,1),
                                'vram_peak_mb':   round(s1.peak,1),
                                'vram_after_mb':  round(s1.vram_after,1),
                                'time_sec':        round(s1.elapsed,3)})

        # Audio features (subprocess — VRAM not tracked)
        t0        = time.perf_counter()
        feat_proc = get_audio_features(wav)
        feat_time = time.perf_counter() - t0
        monitor_records.append({'patient': pid, 'pipeline': '2a. AudioFeat (subprocess)',
                                'vram_before_mb': None, 'vram_peak_mb': None,
                                'vram_after_mb':  None, 'time_sec': round(feat_time,3)})

        # Pipeline 2 — Wav2Vec2 embedding
        with mon.measure('2b. Wav2Vec2 embedding') as s2:
            emb_proc = get_embeddings(wav)
        monitor_records.append({'patient': pid, 'pipeline': '2b. Wav2Vec2 embedding',
                                'vram_before_mb': round(s2.vram_before,1),
                                'vram_peak_mb':   round(s2.peak,1),
                                'vram_after_mb':  round(s2.vram_after,1),
                                'time_sec':        round(s2.elapsed,3)})

        # Pipeline 3 — NeuroXVocal classification
        with mon.measure('3. NeuroXVocal classify') as s3:
            pred, conf = predict(text, feat_proc, emb_proc)
        monitor_records.append({'patient': pid, 'pipeline': '3. NeuroXVocal classify',
                                'vram_before_mb': round(s3.vram_before,1),
                                'vram_peak_mb':   round(s3.peak,1),
                                'vram_after_mb':  round(s3.vram_after,1),
                                'time_sec':        round(s3.elapsed,3)})

        results.append({'patient_id': pid, 'prediction': pred,
                        'label': 'AD' if pred==1 else 'CN',
                        'confidence': round(conf,4),
                        'transcription': text[:100]})

    except Exception as e:
        print(f'ERROR {pid}: {e}')
        results.append({'patient_id': pid, 'prediction': None,
                        'label': 'ERROR', 'confidence': None, 'transcription': str(e)})

df_results = pd.DataFrame(results)
df_monitor = pd.DataFrame(monitor_records)

df_results.to_csv(OUTPUT_CSV,  index=False)
df_monitor.to_csv(MONITOR_CSV, index=False)
print(f'\nPredictions saved → {OUTPUT_CSV}')
print(f'VRAM log saved    → {MONITOR_CSV}')
df_results[['patient_id','label','confidence']]

## 7. Prediction Summary

In [ ]:
valid = df_results.dropna(subset=['prediction'])
print(f'Total processed : {len(valid)} / {len(df_results)}')
print(f'Predicted AD    : {(valid["prediction"]==1).sum()}')
print(f'Predicted CN    : {(valid["prediction"]==0).sum()}')
print(f'\nConfidence stats:')
print(valid['confidence'].describe().round(4))

## 8. VRAM & Timing Report

Peak and minimum VRAM across all patients, per pipeline.

In [ ]:
import pandas as pd

# ── Model load table ─────────────────────────────────────────────────
df_load = pd.DataFrame(mon.load_records)
print('MODEL LOAD — VRAM COST')
print('=' * 62)
print(df_load.to_string(index=False))
print()

# ── Per-pipeline summary ─────────────────────────────────────────────
tracked = df_monitor[df_monitor['vram_peak_mb'].notna()].copy()

summary = tracked.groupby('pipeline').agg(
    peak_max  = ('vram_peak_mb', 'max'),
    peak_min  = ('vram_peak_mb', 'min'),
    peak_mean = ('vram_peak_mb', 'mean'),
    time_mean = ('time_sec',     'mean'),
    time_total= ('time_sec',     'sum'),
).round(2)

print('VRAM & TIMING SUMMARY — per pipeline (across all patients)')
print('=' * 72)
print(f'{"Pipeline":<28}  {"Peak max":>9}  {"Peak min":>9}  {"Peak avg":>9}  {"Avg time":>9}')
print('-' * 72)
for pipe, row in summary.iterrows():
    print(f'{pipe:<28}  {row.peak_max:>8.1f}M  {row.peak_min:>8.1f}M  {row.peak_mean:>8.1f}M  {row.time_mean:>8.2f}s')
print('=' * 72)
print(f'Global peak VRAM (across all steps) : {mon.global_peak_mb():.1f} MB')
print(f'Global min  VRAM (lowest peak seen) : {tracked["vram_peak_mb"].min():.1f} MB')

# ── Full per-patient table ───────────────────────────────────────────
print('\nPER-PATIENT DETAIL')
print('=' * 80)
print(df_monitor.to_string(index=False))

## 9. Accuracy vs Ground Truth (optional)

In [ ]:
if LABELS_CSV is not None:
    gt     = pd.read_csv(LABELS_CSV)
    merged = df_results.merge(gt, on='patient_id', suffixes=('_pred','_true'))
    merged = merged.dropna(subset=['prediction'])
    y_true = merged['label_true'].astype(int).values
    y_pred = merged['prediction'].astype(int).values
    acc    = accuracy_score(y_true, y_pred)
    print(f'Test Accuracy: {acc*100:.2f}%  ({int(acc*len(y_true))}/{len(y_true)})\n')
    print(classification_report(y_true, y_pred, target_names=['CN','AD']))
    print('Confusion Matrix:')
    print(pd.DataFrame(confusion_matrix(y_true, y_pred),
                       index=['True CN','True AD'], columns=['Pred CN','Pred AD']))
else:
    print('No labels set. Set LABELS_CSV in Cell 3 to compute accuracy.')

## 10. Download Results

In [ ]:
from google.colab import files
files.download(OUTPUT_CSV)
files.download(MONITOR_CSV)